In [2]:
class Node:
    def __init__(self, parent, label):
        self.parent = parent
        self.label = label
        self.children = []
        #self.depth = 0 if parent is None else parent.depth + 1

    def set_child(self, child):
        child.parent = self
        #child.depth = self.depth + 1
        self.children.append(child)

    def set_depth(self, depth):
        self.depth = depth
        for child in self.children:
            child.set_depth(depth + 1)

    def get_max_depth(self): 
        self.max_depth = max([self.depth, *[child.get_max_depth() for child in self.children]])
        return self.max_depth

    def set_dfs(self, dfs):
        self.dfs = {}
        for key in dfs:
            df = dfs[key]
            if self.label == "Not relevant":
                self.dfs[key] = df[df["predicted_tag"].str.contains("'"+self.parent.label+"', 'Not relevant'")] # Make sure its the ones where this branch is not relevant
            else:
                self.dfs[key] = df[df["predicted_tag"].str.contains("'"+self.label+"'")]
        

        for child in self.children:
            child.set_dfs(self.dfs)
    
    def print_info(self, fnc):
        try:
            print("\t"*self.depth, fnc(self))
        except StopIteration:
            return
        for child in self.children:
            child.print_info(fnc)

In [3]:
from metadata_schemas.ai_taxonomy import AI_TAXONOMY


In [4]:
root = Node(None, 'AI Taxonomy')

def populate(node, branch):
    if isinstance(branch, dict):
        for label, subtree in branch.items():
            child = Node(node, label)
            node.set_child(child)
            populate(child, subtree)
    elif isinstance(branch, list):
        for item in branch:
            if isinstance(item, dict):
                for label, subtree in item.items():
                    child = node.set_child(Node(node, label))
                    populate(child, subtree)
            else:
                node.set_child(Node(node, item))
    elif branch is None:
        return
    else:
        node.set_child(Node(node, branch))

populate(root, AI_TAXONOMY)
root.set_depth(-1)



In [5]:
import os
import pandas as pd
all_dfs = {}
for file in os.listdir("trend_csvs"):
    df = pd.read_csv("trend_csvs/"+file, index_col=0)
    all_dfs[file[:-15]] = df
    df["date"] = pd.to_datetime(df["date"])
    df["predicted_tag"] = df["predicted_tag"].astype(str)


In [6]:
# Force set df for root node (sinc its label is not in the dfs.)
root.dfs = all_dfs
for child in root.children:
    child.set_dfs(all_dfs)

/tmp/ipykernel_4170317/1133004729.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  self.dfs[key] = df[df["predicted_tag"].str.contains("'"+self.label+"'")]


In [7]:
def name_and_sizes(node):
    return node.label + "\t"*2 + str([[key, len(df)] for key, df in node.dfs.items()])


def fraction_printer(node):
    string = node.label + "\t\t"
    fractions = []
    for key in ["arx_long", "hf", "dlw_long"]:
        fraction = (100*len(node.dfs[key]))//len(all_dfs[key])
        fractions.append(fraction)
        string+= key+": "+str(fraction)+"   "
    if sum(fractions)<10:
        raise StopIteration
    return string
    

In [8]:
root.print_info(fraction_printer)

 AI Taxonomy		arx_long: 100   hf: 100   dlw_long: 100   
 Model architecture		arx_long: 100   hf: 100   dlw_long: 100   
	 Not relevant		arx_long: 25   hf: 1   dlw_long: 15   
	 Neural/deep learning architectures		arx_long: 70   hf: 95   dlw_long: 84   
		 Feed-forward networks		arx_long: 3   hf: 15   dlw_long: 8   
			 Multilayer perceptrons		arx_long: 2   hf: 14   dlw_long: 8   
		 Convolutional networks		arx_long: 21   hf: 8   dlw_long: 17   
			 2D CNNs		arx_long: 9   hf: 3   dlw_long: 5   
		 Sequence models		arx_long: 22   hf: 55   dlw_long: 31   
			 Recurrent neural networks		arx_long: 10   hf: 10   dlw_long: 12   
			 Transformer architectures		arx_long: 9   hf: 44   dlw_long: 17   
				 Encoder-decoder transformers		arx_long: 3   hf: 20   dlw_long: 4   
				 Encoder-only transformers		arx_long: 0   hf: 13   dlw_long: 1   
					 BERT		arx_long: 0   hf: 12   dlw_long: 1   
		 Graph and relational		arx_long: 7   hf: 2   dlw_long: 5   
		 Physics-inspired and continuous models		a

In [9]:
def other_printer(node):
    raise NotImplementedError # interesting idea but i need to disregard "Other" from other taxonomies
    string = node.label + "\t\t"
    fractions = []
    for key in ["arx_long", "hf", "dlw_long"]:
        fraction = (100*len(node.dfs[key][node.dfs[key]["predicted_tag"].str.contains("Other")]))//len(all_dfs[key])
        fractions.append(fraction)
        string+= key+": "+str(fraction)+"   "
    if sum(fractions)<3:
        raise StopIteration
    return string

In [11]:
root.get_max_depth()

6

In [12]:
def calc_absolute_falloffs(self):
    """Absolute falloff is number of documents labeled with this node but not with any child node"""
    continuers = {}
    for key in self.dfs:
        continuers[key] = 0
        for child in self.children:
            continuers[key] += len(child.dfs[key])
    
    falloffs = {}
    for key in self.dfs:
        falloffs[key] = len(self.dfs[key]) - continuers[key]
    self.absolute_falloffs = falloffs
    


def calc_absolute_depth_score(self):
    """Absolute depth score is sum(depth_datio*absolute_falloff), summed over all descendants"""

    # Depth ratio is how deep the node is relative to max depth under this node
    depth_ratio = self.depth / self.max_depth

    # calc absolute depth score from falloffs of this node
    calc_absolute_falloffs(self)
    abs_depth_scores = {} 
    for key in self.dfs:
        abs_depth_scores[key] = self.absolute_falloffs[key] * depth_ratio
    
    # add abs_depth_score from children
    for child in self.children:
        calc_absolute_depth_score(child)
        for key in self.dfs:
            abs_depth_scores[key] += child.abs_depth_scores[key]
        
    self.abs_depth_scores = abs_depth_scores

calc_absolute_depth_score(root)

In [18]:
def falloff_ratio_printer(node):
    string = node.label + "\t\t"
    for key in ["arx_long", "hf", "dlw_long"]:
        falloff_ratio = (100*node.absolute_falloffs[key])//len(all_dfs[key])
        string+= key+": "+str(falloff_ratio)+"   "
    return string

#root.print_info(falloff_ratio_printer)

In [35]:
def avg_depth_ratio_printer(node):
    if len(node.children) == 0:
        raise StopIteration # These have avg depth == 100 and are not interesting
    string = node.label + "\t\t"
    lens = []
    for key in ["arx_long", "hf", "dlw_long"]:
        if len(node.dfs[key]) < 5:
            avg_depth = "-"
        else:
            avg_depth = int(100*node.abs_depth_scores[key])//len(node.dfs[key])
        lens.append(len(node.dfs[key]))
        string+= key+": "+str(avg_depth)+"   "
    if sum(lens) < 10:
        raise StopIteration
    return string

root.print_info(avg_depth_ratio_printer)
    

 AI Taxonomy		arx_long: 407   hf: 406   dlw_long: 411   
 Model architecture		arx_long: 84   hf: 84   dlw_long: 82   
	 Classical machine learning models		arx_long: 85   hf: 92   dlw_long: -   
	 Neural/deep learning architectures		arx_long: 78   hf: 84   dlw_long: 78   
		 Feed-forward networks		arx_long: 97   hf: 98   dlw_long: 99   
		 Convolutional networks		arx_long: 94   hf: 94   dlw_long: 92   
		 Sequence models		arx_long: 78   hf: 92   dlw_long: 82   
			 Recurrent neural networks		arx_long: 75   hf: 76   dlw_long: 75   
			 Transformer architectures		arx_long: 83   hf: 96   dlw_long: 88   
				 Encoder-only transformers		arx_long: 97   hf: 99   dlw_long: 96   
				 Multimodal transformers		arx_long: 98   hf: -   dlw_long: 96   
		 Graph and relational		arx_long: 93   hf: 92   dlw_long: 91   
		 Physics-inspired and continuous models		arx_long: 92   hf: 88   dlw_long: 89   
		 Neuro-symbolic models		arx_long: 97   hf: -   dlw_long: 94   
 AI problem type		arx_long: 95   hf: 92